# TP-2: CleaningData - Applying OOP and Design Patterns

## Exercise 2: Apply Pattern for Missing Values Handling (Strategy Pattern)

**Objective:**
- Learn the **Strategy Pattern** - one interface, many interchangeable behaviors.
- Practice **inheritance** and **polymorphism**.

**Concept:** We often need different strategies to handle missing data (drop, fill mean, fill mode, ...).
The pattern allows flexible switching between methods **without changing the main code**.

**Instructions:**
1. Create an abstract class `MissingValueStrategy` with a method `handle(df)`.
2. Create subclasses: `DropMissing`, `FillMean`, `FillMode`.
3. Create a `DataCleaner` class that applies the chosen strategy.

In [1]:
import os
from abc import ABC, abstractmethod

import pandas as pd

### CSVReader (from Exercise 1)
We reuse the `CSVReader` class to load the data.

In [2]:
class CSVReader:
    """Encapsulates reading and previewing a CSV file."""

    def __init__(self, file_path):
        self.file_path = file_path
        self._data = None  # private: loaded DataFrame

    def read(self):
        """Load the CSV file into a pandas DataFrame and return it."""
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"File not found: {self.file_path}")
        self._data = pd.read_csv(self.file_path)
        return self._data

    def preview(self, n=5):
        """Print the top n rows of the data."""
        if self._data is None:
            self.read()
        print(f"Top {n} rows of '{self.file_path}':")
        print(self._data.head(n))

    @property
    def data(self):
        """Read-only access to the loaded DataFrame."""
        return self._data

    def __repr__(self):
        status = "loaded" if self._data is not None else "not loaded"
        return f"CSVReader(file_path='{self.file_path}', status={status})"

### Step 1: Abstract class `MissingValueStrategy`
- It inherits from `ABC` (Abstract Base Class).
- `@abstractmethod` forces every subclass to implement `handle(df)`.
- It is the **common interface** shared by all strategies; it cannot be used directly.

In [3]:
class MissingValueStrategy(ABC):
    """Interface: every strategy must implement handle(df)."""

    @abstractmethod
    def handle(self, df):
        """Return a new DataFrame with missing values handled."""
        pass

### Step 2: Concrete strategies (inheritance)
Each subclass **inherits** from `MissingValueStrategy` and gives its **own version** of `handle(df)`:

| Strategy | What it does |
|---|---|
| `DropMissing` | Removes rows that contain any `NaN` |
| `FillMean` | Replaces `NaN` in numeric columns with the column mean |
| `FillMode` | Replaces `NaN` in every column with the most frequent value |

Each strategy works on a **copy**, so the original DataFrame is not changed.

In [4]:
class DropMissing(MissingValueStrategy):
    """Remove every row that has at least one missing value."""

    def handle(self, df):
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    """Fill missing values in numeric columns with the column mean."""

    def handle(self, df):
        df = df.copy()
        num_cols = df.select_dtypes(include="number").columns
        df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
        return df


class FillMode(MissingValueStrategy):
    """Fill missing values in every column with its most frequent value."""

    def handle(self, df):
        df = df.copy()
        for col in df.columns:
            if df[col].isnull().any():
                df[col] = df[col].fillna(df[col].mode()[0])
        return df

### Step 3: `DataCleaner` (the context)
`DataCleaner` does not know *how* missing values are handled. It only calls `strategy.handle(df)`.
We can change the strategy at any time with `set_strategy()`, without modifying `DataCleaner`.

In [5]:
class DataCleaner:
    """Context class: applies whichever strategy it is given."""

    def __init__(self, strategy):
        self._strategy = strategy

    def set_strategy(self, strategy):
        """Switch to another strategy at runtime."""
        self._strategy = strategy

    def clean(self, df):
        print(f"Applying strategy: {type(self._strategy).__name__}")
        return self._strategy.handle(df)

### Test with `sample_data.csv`

In [6]:
reader = CSVReader("sample_data.csv")
df = reader.read()
print("Missing values before cleaning:", df.isnull().sum().sum())
df

Missing values before cleaning: 7


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [7]:
# Strategy 1: DropMissing
cleaner = DataCleaner(DropMissing())
dropped = cleaner.clean(df)
print("Shape after drop:", dropped.shape)
dropped

Applying strategy: DropMissing
Shape after drop: (2, 7)


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,5,Eva,27.0,160.0,55.0,New York,88.0


In [8]:
# Strategy 2: FillMean (switch strategy, same cleaner)
cleaner.set_strategy(FillMean())
filled_mean = cleaner.clean(df)
filled_mean

Applying strategy: FillMean


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


In [9]:
# Strategy 3: FillMode
cleaner.set_strategy(FillMode())
filled_mode = cleaner.clean(df)
filled_mode

Applying strategy: FillMode


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


In [10]:
# Polymorphism: same method call handle(df), different behavior
for strategy in [DropMissing(), FillMean(), FillMode()]:
    result = strategy.handle(df)
    print(f"{type(strategy).__name__:<12} -> shape {result.shape}, missing = {result.isnull().sum().sum()}")

DropMissing  -> shape (2, 7), missing = 0
FillMean     -> shape (7, 7), missing = 0
FillMode     -> shape (7, 7), missing = 0


In [11]:
# The abstract class cannot be instantiated
try:
    MissingValueStrategy()
except TypeError as e:
    print("Error:", e)

Error: Can't instantiate abstract class MissingValueStrategy without an implementation for abstract method 'handle'


### Summary
- **Strategy Pattern**: `MissingValueStrategy` is the interface; `DropMissing`, `FillMean`, `FillMode` are interchangeable strategies.
- **Inheritance**: every strategy inherits from `MissingValueStrategy`.
- **Polymorphism**: the same call `handle(df)` behaves differently depending on the object.
- **Flexibility**: `DataCleaner` never changes - we only pass a different strategy.

Observation: `DropMissing` keeps only 2 of 7 rows (we lose a lot of data), while `FillMean` and `FillMode` keep all 7 rows.